# Q13.
```{admonition}
:class: note
Repeat the analysis on the `IMDb` data using a similarly structured neural network. We used 16 hidden units at each of two hidden layers. Explore the effect of increasing this to 32 and 64 units per layer, with and without 30% dropout regularization.

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [3]:
from ISLP.torch.imdb import load_tensor

W0424 11:54:16.631000 164704 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [4]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

In [5]:
(imdb_train,imdb_test) = load_tensor(root='data/IMDB')

In [6]:
torch.manual_seed(1728)

# 32 units, no dropout
class IMDB1(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self._forward = nn.Sequential(
            nn.Linear(input_size,32),
            nn.ReLU(),
            nn.Linear(32,32),
            nn.ReLU(),
            nn.Linear(32,1)
        )

    def forward(self, x):
        return torch.flatten(self._forward(x))

# 64 units, no dropout
class IMDB2(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self._forward = nn.Sequential(
            nn.Linear(input_size,64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,1)
        )

    def forward(self, x):
        return torch.flatten(self._forward(x))

# 32 units, dropout
class IMDB3(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self._forward = nn.Sequential(
            nn.Linear(input_size,32),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(32,32),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(32,1)
        )

    def forward(self, x):
        return torch.flatten(self._forward(x))

# 64 units, dropout
class IMDB4(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self._forward = nn.Sequential(
            nn.Linear(input_size,64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64,1)
        )

    def forward(self, x):
        return torch.flatten(self._forward(x))

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_loader = DataLoader(
    imdb_train,
    batch_size=64,
    shuffle=True
)

X_test_t = imdb_test.tensors[0].to(device)
y_test = imdb_test.tensors[1]

In [8]:
imdb_model1 = IMDB1(imdb_train.tensors[0].shape[1]).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(imdb_model1.parameters(),lr=0.001)
epochs = 30

for epoch in range(epochs):
    imdb_model1.train()

    for X_batch, y_batch in train_loader:
        mses = imdb_model1(X_batch.to(device))
        loss = criterion(mses,y_batch.to(device))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [9]:
imdb_model1.eval()

with torch.no_grad():
    pred = torch.sigmoid(imdb_model1(X_test_t.to(device))) > 0.5
y_pred = pred.cpu().numpy()

print(classification_report(y_test,y_pred,digits=4))

              precision    recall  f1-score   support

         0.0     0.8534    0.8599    0.8567     12500
         1.0     0.8588    0.8523    0.8556     12500

    accuracy                         0.8561     25000
   macro avg     0.8561    0.8561    0.8561     25000
weighted avg     0.8561    0.8561    0.8561     25000



In [10]:
imdb_model2 = IMDB2(imdb_train.tensors[0].shape[1]).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(imdb_model2.parameters(),lr=0.001)
epochs = 30

for epoch in range(epochs):
    imdb_model2.train()

    for X_batch, y_batch in train_loader:
        mses = imdb_model2(X_batch.to(device))
        loss = criterion(mses,y_batch.to(device))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [11]:
imdb_model2.eval()

with torch.no_grad():
    pred = torch.sigmoid(imdb_model2(X_test_t.to(device))) > 0.5
y_pred = pred.cpu().numpy()

print(classification_report(y_test,y_pred,digits=4))

              precision    recall  f1-score   support

         0.0     0.8583    0.8673    0.8628     12500
         1.0     0.8659    0.8568    0.8613     12500

    accuracy                         0.8620     25000
   macro avg     0.8621    0.8620    0.8620     25000
weighted avg     0.8621    0.8620    0.8620     25000



In [12]:
imdb_model3 = IMDB3(imdb_train.tensors[0].shape[1]).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(imdb_model3.parameters(),lr=0.001)
epochs = 30

for epoch in range(epochs):
    imdb_model3.train()

    for X_batch, y_batch in train_loader:
        mses = imdb_model3(X_batch.to(device))
        loss = criterion(mses,y_batch.to(device))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [13]:
imdb_model3.eval()

with torch.no_grad():
    pred = torch.sigmoid(imdb_model3(X_test_t.to(device))) > 0.5
y_pred = pred.cpu().numpy()

print(classification_report(y_test,y_pred,digits=4))

              precision    recall  f1-score   support

         0.0     0.8604    0.8838    0.8719     12500
         1.0     0.8806    0.8566    0.8684     12500

    accuracy                         0.8702     25000
   macro avg     0.8705    0.8702    0.8702     25000
weighted avg     0.8705    0.8702    0.8702     25000



In [14]:
imdb_model4 = IMDB4(imdb_train.tensors[0].shape[1]).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(imdb_model4.parameters(),lr=0.001)
epochs = 30

for epoch in range(epochs):
    imdb_model4.train()

    for X_batch, y_batch in train_loader:
        mses = imdb_model4(X_batch.to(device))
        loss = criterion(mses,y_batch.to(device))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [15]:
imdb_model4.eval()

with torch.no_grad():
    pred = torch.sigmoid(imdb_model4(X_test_t.to(device))) > 0.5
y_pred = pred.cpu().numpy()

print(classification_report(y_test,y_pred,digits=4))

              precision    recall  f1-score   support

         0.0     0.8740    0.8704    0.8722     12500
         1.0     0.8709    0.8745    0.8727     12500

    accuracy                         0.8724     25000
   macro avg     0.8724    0.8724    0.8724     25000
weighted avg     0.8724    0.8724    0.8724     25000

